In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
base_folder = "/content/drive/MyDrive/Colab Notebooks/heart_diagnosis_fall2025"
%cd "{base_folder}"

/content/drive/MyDrive/Colab Notebooks/heart_diagnosis_fall2025


In [3]:
import sqlite3
import pandas as pd

# base_folder should be defined in your environment (e.g., base_folder = ".")
conn = sqlite3.connect(f"{base_folder}/data/heart.db")

heart_data = pd.read_sql_query(
    """
    SELECT
        p.patient_id,
        p.age,
        p.sex,
        p.cp_id,
        p.trestbps,
        p.chol,
        p.fbs,
        p.ecg_id,
        p.thalach,
        p.exang,
        p.slope_id,
        p.ca,
        p.thal_id,
        d.target
    FROM patient AS p
    JOIN diagnosis AS d
        ON p.patient_id = d.patient_id
    ORDER BY p.patient_id
    """,
    conn,
)
conn.close()

# Display the first few rows to verify oldpeak is gone and target is joined
heart_data.head()

,patient_id,age,sex,cp_id,trestbps,chol,fbs,ecg_id,thalach,exang,slope_id,ca,thal_id,target
0,0,52,1,1,125,212,0,1,168,0,1,2,1,0
1,1,53,1,1,140,203,1,2,155,1,2,0,1,0
2,2,70,1,1,145,174,0,1,125,1,2,0,1,0
3,3,61,1,1,148,203,0,1,161,0,1,1,1,0
4,4,62,0,1,138,294,1,1,106,0,3,3,2,0


In [4]:
import json

print("=" * 80)
print("ANALYZING HEART DATA FOR STREAMLIT APP")
print("=" * 80)

# Based on your simplified model requirements
# We only need schema data for the features the model actually sees
numerical_features = ['ca']
categorical_features = ['cp_id', 'thal_id']

# Create schema dictionary
data_schema = {
    "numerical": {},
    "categorical": {}
}

# Analyze numerical features
print("\n" + "-" * 80)
print("NUMERICAL FEATURES")
print("-" * 80)
print(f"{'Feature':<25} {'Min':<15} {'Max':<15} {'Mean':<15} {'Median':<15}")
print("-" * 80)

for feature in numerical_features:
    # Ensure we use the heart_data dataframe loaded from SQL
    min_val = float(heart_data[feature].min())
    max_val = float(heart_data[feature].max())
    mean_val = float(heart_data[feature].mean())
    median_val = float(heart_data[feature].median())

    data_schema["numerical"][feature] = {
        "min": min_val,
        "max": max_val,
        "mean": mean_val,
        "median": median_val
    }

    print(f"{feature:<25} {min_val:<15.2f} {max_val:<15.2f} {mean_val:<15.2f} {median_val:<15.2f}")

# Analyze categorical features
print("\n" + "-" * 80)
print("CATEGORICAL FEATURES")
print("-" * 80)

for feature in categorical_features:
    # Convert unique values to a sorted list for cleaner Streamlit dropdowns
    unique_values = sorted(heart_data[feature].unique().tolist())
    value_counts = heart_data[feature].value_counts().to_dict()

    data_schema["categorical"][feature] = {
        "unique_values": unique_values,
        "value_counts": value_counts
    }

    print(f"\n{feature}:")
    print(f"  Unique values: {unique_values}")
    print(f"  Value counts:")
    for value, count in value_counts.items():
        print(f"    {value}: {count} ({count/len(heart_data)*100:.1f}%)")

# Save schema to JSON file
# Using your base_folder variable defined earlier in your notebook
output_file = f"{base_folder}/data/data_schema.json"
with open(output_file, 'w') as f:
    json.dump(data_schema, f, indent=2)

print("\n" + "=" * 80)
print(f"✓ Data schema saved to {output_file}")
print("=" * 80)

# Display the JSON structure
print("\n" + "-" * 80)
print("GENERATED SCHEMA (data_schema.json)")
print("-" * 80)
print(json.dumps(data_schema, indent=2))

ANALYZING HEART DATA FOR STREAMLIT APP

--------------------------------------------------------------------------------
NUMERICAL FEATURES
--------------------------------------------------------------------------------
Feature                   Min             Max             Mean            Median         
--------------------------------------------------------------------------------
ca                        0.00            4.00            0.75            0.00           

--------------------------------------------------------------------------------
CATEGORICAL FEATURES
--------------------------------------------------------------------------------

cp_id:
  Unique values: [1, 2, 3, 4]
  Value counts:
    1: 497 (48.5%)
    3: 284 (27.7%)
    2: 167 (16.3%)
    4: 77 (7.5%)

thal_id:
  Unique values: [1, 2, 3, 4]
  Value counts:
    2: 544 (53.1%)
    1: 410 (40.0%)
    3: 64 (6.2%)
    4: 7 (0.7%)

✓ Data schema saved to /content/drive/MyDrive/Colab Notebooks/heart_diagnosis_